# 📊 Stats Wales Data Explorer

Welcome to the Stats Wales workshop notebook! 🏴󠁧󠁢󠁷󠁬󠁳󠁿

This notebook is **ready to go** — just run each cell from top to bottom. The only thing you need to do is pick a dataset and paste its ID!

### What you'll do:
1. ✅ Load all the Python libraries (just run it)
2. 🔍 Browse the list of available Stats Wales datasets
3. ✏️ **Pick a dataset and paste its ID** into the code
4. 📊 Create a bar chart (customise the title & colours)
5. 💾 Export everything for your GitHub Pages dashboard

### Your focus:
- **Choose a dataset** that interests you
- **Customise** small bits marked with ✏️ (title, colours, etc.)
- **Commit your changes** using Git — that's the real skill here!

---

**💡 Tip:** Use GitHub Copilot to help you! Press `Ctrl+I` (or `Cmd+I` on Mac) to open Copilot Chat. Try prompts like:
- *"Explain what this code does"*
- *"Help me change the chart colours"*

## Step 1 — Load Libraries

Run this cell first — it loads everything you need.
*(Everything is already installed in Codespaces — no setup required!)*

In [ ]:
import requests
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import StringIO

print("✅ Libraries loaded successfully! Move on to Step 2.")

## Step 2 — Browse Available Datasets

Run this cell to see what datasets are available on Stats Wales. Scroll through the list and **find one that interests you** — you'll need its `id` in the next step.

> **💡 Tip:** Look for something relevant to your work — health, population, education, economics. You'll be more engaged if the data means something to you!

In [ ]:
# ─── Browse all available Stats Wales datasets ────────────────────────
datasets_url = "https://api.stats.gov.wales/v1/?lang=en-gb&page_number=1&page_size=100"

response = requests.get(datasets_url)
response.raise_for_status()

datasets = response.json()
df_datasets = pd.DataFrame(datasets.get("value", datasets))

print(f"📋 Found {len(df_datasets)} datasets\n")
print("Scroll through the list below and pick one you like.")
print("Copy its 'id' value — you'll paste it in the next step!\n")

# Show the id and title so you can pick one
cols_to_show = [c for c in ["id", "title", "description"] if c in df_datasets.columns]
if cols_to_show:
    display(df_datasets[cols_to_show])
else:
    display(df_datasets)

## Step 3 — Fetch Your Chosen Dataset

Now paste the `id` you copied from Step 2 into the `DATASET_ID` line below, replacing the placeholder text. Then run the cell!

**How to do it:**
1. Find the `DATASET_ID = "PASTE_YOUR_DATASET_ID_HERE"` line below
2. Delete `PASTE_YOUR_DATASET_ID_HERE` and paste in the `id` you copied (keep the quotes!)
3. Run the cell with **Shift+Enter**

> **💡 Copilot tip:** If you get an error, ask Copilot: *"I'm getting this error: [paste error]. How do I fix it?"*

In [ ]:
# ─── Fetch data from your chosen dataset ──────────────────────────────
# ✏️ PASTE YOUR DATASET ID BELOW (replace the placeholder text)
DATASET_ID = "PASTE_YOUR_DATASET_ID_HERE"
# ──────────────────────────────────────────────────────────────────────

if DATASET_ID == "PASTE_YOUR_DATASET_ID_HERE":
    print("⚠️  You need to paste a Dataset ID first!")
    print("   Go back to Step 2, pick a dataset, and copy its 'id' value.")
    print("   Then replace PASTE_YOUR_DATASET_ID_HERE above (keep the quotes).")
else:
    # Fetch the data
    params = {
        "lang": "en-gb",
        "page_number": 1,
        "page_size": 100,
    }

    data_url = f"https://api.stats.gov.wales/v1/{DATASET_ID}/view"
    response = requests.get(data_url, params=params)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data.get("value", data))

    print(f"✅ Loaded {len(df)} rows and {len(df.columns)} columns from Stats Wales")
    print(f"\nColumns: {list(df.columns)}\n")
    display(df.head(10))

## Step 4 — Create Your Chart 📊

This cell creates a bar chart from the data. It automatically detects the right columns!

### ✏️ Things you can customise (try changing these!):
- **`CHART_TITLE`** — Give your chart a meaningful name
- **`BAR_COLOURS`** — Change the colours (use hex codes like `"#007C45"`)
- **`Y_LABEL`** — Change the Y-axis label

> **💡 Copilot tip:** Highlight the code and ask: *"Change this to a horizontal bar chart"* or *"Add data labels on each bar"*

In [ ]:
# ─── Create a chart ────────────────────────────────────────────────────
# ✏️ CUSTOMISE THESE — change the title, colours, or labels!
CHART_TITLE = "NAME OF YOUR CHART"
Y_LABEL = "Value"
BAR_COLOURS = ["#007C45", "#D40000"]  # Welsh green and red 🏴󠁧󠁢󠁷󠁬󠁳󠁿
# ──────────────────────────────────────────────────────────────────────

# Automatically detect the best columns for the chart
text_cols = df.select_dtypes(include="object").columns.tolist()
CATEGORY_COLUMN = text_cols[0] if text_cols else df.columns[0]

# Find the numeric value column
VALUE_COLUMN = None
for col in df.columns:
    numeric_vals = pd.to_numeric(df[col], errors="coerce")
    if numeric_vals.notna().sum() > 0 and col != CATEGORY_COLUMN:
        VALUE_COLUMN = col
        break

if VALUE_COLUMN is None:
    VALUE_COLUMN = df.columns[-1]

print(f"📊 Charting: {VALUE_COLUMN} by {CATEGORY_COLUMN}\n")

# Prepare the data
plot_df = df.copy()
plot_df[VALUE_COLUMN] = pd.to_numeric(plot_df[VALUE_COLUMN], errors="coerce")
chart_data = plot_df.groupby(CATEGORY_COLUMN)[VALUE_COLUMN].sum().sort_values(ascending=False)

# Create the chart
fig, ax = plt.subplots(figsize=(10, 6))
colours = (BAR_COLOURS * (len(chart_data) // len(BAR_COLOURS) + 1))[:len(chart_data)]
bars = ax.bar(chart_data.index, chart_data.values, color=colours)

# Add data labels on each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{height:,.0f}', ha='center', va='bottom', fontsize=10)

ax.set_title(CHART_TITLE, fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel(CATEGORY_COLUMN, fontsize=12)
ax.set_ylabel(Y_LABEL, fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=45, ha="right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

# Save chart for the dashboard
plt.savefig("../docs/chart.png", dpi=150, bbox_inches="tight")
print("✅ Chart saved to docs/chart.png")

plt.show()

## Step 5 — Export Data for the Dashboard

This cell saves the data as JSON so the HTML dashboard page can use it. Just run it!

In [ ]:
# ─── Export data for the HTML dashboard ────────────────────────────────
export_df = df.copy()

# Clean up numeric columns
for col in export_df.columns:
    export_df[col] = pd.to_numeric(export_df[col], errors="ignore")

output_path = "../docs/data.json"
export_df.to_json(output_path, orient="records", indent=2)

print(f"✅ Data exported to docs/data.json")
print(f"   {len(export_df)} records, {len(export_df.columns)} fields\n")
print("First record preview:")
print(json.dumps(export_df.iloc[0].to_dict(), indent=2, default=str))

## 🎉 Done! Now head back to the Workshop Guide

You've browsed Stats Wales datasets, picked one, created a chart, and exported everything for the dashboard!

### 👉 Next: Continue with the [Workshop Guide](../WORKSHOP_GUIDE.md)

Open **`WORKSHOP_GUIDE.md`** and pick up from **Module 4.4 — Commit your changes**. The guide will walk you through:

- **Committing** your work using Git (Module 4.4)
- **Publishing** your dashboard with GitHub Pages (Module 5)
- **Sharing** your work with colleagues (Module 6)

> 💡 The Workshop Guide has detailed, step-by-step instructions for each of these — follow it rather than trying to figure it out on your own!